# Residual condicionado y ciclo único del remolque

## tl;dr
La curva histórica US produce retención bruta de 58.24% en un escenario de tenencia planificada de 36 meses y 82,677 millas/año. No es vida restante medida, residual neto ni calibración 2026. El stress sintético del remolque demuestra que calcular max por tramo puede sumar 192 frente a 125 del ciclo único.

## Context & Methods
Compañero reproducible de B2_USED_RESIDUAL_SHARED_CYCLE_2026-09-29.json. Consulta 2026-09-29, HEAD 4f5fc2d. Se reutilizan el perfil B2 y el cálculo de remolque compartido; no cambios de motor ni adopción.

### Key Assumptions
36 meses es un horizonte analítico de tenencia desde adquisición, inspirado por el caso de primer propietario de Argonne, no vida técnica/económica restante de esta unidad usada. La combinación de modelo de anuncios 2019 y exposición nacional 2024 es condicional, no cohorte observada ni forecast. ±20% es stress de uso, no intervalo publicado. La retención proporcional se ancla al precio pedido 45,000 sin certificar moneda ni tratarlo como adquisición capitalizable. A y K siguen desconocidos.

## Data
Fuentes y sus límites están en el JSON: ANL/ESD-21/4 Eq 3.8 y tabla 3.11, impreso 58-60 / PDF 89-91; ventana primera tenencia de 3 años impreso 170 / PDF 201; exposición ATRI 2024 reutilizada. PDF público obtenido de regulations.gov, hash c49755ae06436d51fda2c629d9b3ab8595279713f7e852d825e9a90e400d4425. Se revisaron visualmente páginas relevantes. No se aplican coeficientes US a MX o al remolque.

### 1. Cargar expediente sin escribir bases
Ejecutar desde la raíz del worktree; sólo biblioteca estándar, sin red o credenciales.

In [1]:
import json, math, hashlib
from pathlib import Path

research = Path("docs/research")
d = json.loads((research / "B2_USED_RESIDUAL_SHARED_CYCLE_2026-09-29.json").read_text(encoding="utf-8"))
assert d["calibrated"] is False and d["adoptedValues"] == 0
for item in d["unchangedFiles"]:
    assert hashlib.sha256((research / item["file"]).read_bytes()).hexdigest() == item["sha256"]
print("PASS: JSON, zero adoption and four unchanged prior dossiers")
print("ANL observations: September 2019; publication 2021. Conditional holding: 36 months, not measured remaining life.")


PASS: JSON, zero adoption and four unchanged prior dossiers
ANL observations: September 2019; publication 2021. Conditional holding: 36 months, not measured remaining life.


## Results

### 2. Retención bruta condicionada
q = 0.9071^h × 0.9990^(millas adicionales / 1000). El cociente cancela C y edad/millas iniciales, siempre que conserve la forma proporcional histórica. No confirma que el anuncio sea una predicción sin sesgo para este VIN.

R_neto = V_actual × q − K_disposición. D_mensual = (A_capitalizable − V_actual × q + K_disposición) / N. Las columnas monetarias sólo son escala del anuncio; no son residual neto, interés ni CFU. Tabla de tres escenarios suficiente; no se dibuja una banda que podría confundirse con intervalo estadístico.

In [2]:
# Dimensionless residual ratio: historical proportional model, not a predicted net sale price.
age_factor = d["sources"][0]["annualAgeRetention"]
mile_factor = d["sources"][0]["retentionPerThousandAdditionalMiles"]
checks = 0
print("Scenario | Additional miles | Gross retention % | Gross asking-anchor units | Value loss per 100 anchor/month")
for row in d["usTractor"]["residualScenarios"]:
    years = row["horizonMonths"] / 12
    miles = 82677 * row["multiplier"] * years
    q = math.exp(math.log(age_factor) * years + math.log(mile_factor) * miles / 1000)
    loss_per_100 = 100 * (1 - q) / row["horizonMonths"]
    assert math.isclose(q, row["grossRetentionFraction"], abs_tol=1e-12)
    assert math.isclose(45000*q, row["grossOfferAnchorUnits"], abs_tol=1e-7)
    assert math.isclose(loss_per_100, row["valueLossPer100AnchorPerMonth"], abs_tol=1e-12)
    assert row["netResidualUsd"] is None and row["monthlyDepreciationUsd"] is None
    assert row["remainingEconomicLifeMonths"] is None
    checks += 5
    label = {0.8:"Less use stress",1:"Historical exposure",1.2:"More use stress"}[row["multiplier"]]
    print(f"{label} | {miles:,.1f} | {100*q:.2f}% | {45000*q:,.2f} | {loss_per_100:.4f}")
print("Gross anchor units have no certified currency and are not capitalizable cost or net residual.")
print("Residual scenario spread is not a statistical interval.")
print("Checks:", checks)


Scenario | Additional miles | Gross retention % | Gross asking-anchor units | Value loss per 100 anchor/month
Less use stress | 198,424.8 | 61.20% | 27,539.76 | 1.0778
Historical exposure | 248,031.0 | 58.24% | 26,206.29 | 1.1601
More use stress | 297,637.2 | 55.42% | 24,937.40 | 1.2384
Gross anchor units have no certified currency and are not capitalizable cost or net residual.
Residual scenario spread is not a statistical interval.
Checks: 15


### 3. Conservar un ciclo físico de remolque
Se verifica algebraicamente con fixture sintético lo que pruebas TypeScript comprueban mediante calculateSharedTrailer: calcular el ciclo completo una vez y sólo luego atribuirlo. Las cifras y el reparto 40/60 de este check no son supuestos públicos del piloto. Responsable, país económico y reparto reales permanecen null.

In [3]:
# Independent synthetic allocation check, not a public benchmark, service or settlement.
s = d["sharedTrailer"]["stressOnly"]
c, km_month, days_month = s["monthlyFixedUsd"], s["monthlyTrailerKm"], s["monthlyTrailerDays"]
km = s["mx"]["km"] + s["us"]["km"]
days = s["mx"]["days"] + s["us"]["days"]
whole = max(c * km / km_month, c * days / days_month)
per_leg = sum(max(c * leg["km"]/km_month, c * leg["days"]/days_month) for leg in [s["mx"],s["us"]])
assert whole == s["oneCycleCfuUsd"] == 125
assert per_leg == s["sumOfPerLegMaxUsd"] == 192
assert per_leg-whole == s["overAllocationUsd"] == 67
split = s["splitExample"]
assert split["mxShare"]+split["usShare"] == 1
assert whole*split["mxShare"] == split["mxUsd"] == 50
assert whole*split["usShare"] == split["usUsd"] == 75
assert split["mxUsd"]+split["usUsd"] == whole
assert d["sharedTrailer"]["chosenSettlementShares"] is None
assert d["mxTractor"]["conditionalRetentionFraction"] is None
assert all(value is None for value in d["outputs"].values())
print("Synthetic one-cycle allocation: 125 USD; incorrect sum of two leg-max allocations: 192 USD; excess: 67 USD.")
print("Synthetic post-allocation split: 50 + 75 = 125 USD. The real settlement remains unselected.")
print("PASS: 10 allocation/output checks; MX residual and all economic outputs remain unknown.")


Synthetic one-cycle allocation: 125 USD; incorrect sum of two leg-max allocations: 192 USD; excess: 67 USD.
Synthetic post-allocation split: 50 + 75 = 125 USD. The real settlement remains unselected.
PASS: 10 allocation/output checks; MX residual and all economic outputs remain unknown.


## Takeaways

El residual condicionado US ya tiene una relación pública edad/millas, no un porcentaje arbitrario del precio de un usado. No se confunde horizonte de planificación con supervivencia, vida fiscal o vida de activo nuevo. No hay intervalo de predicción reconstruido ni neto de venta disponible. MX permanece faltante.

No cobrar completo el remolque en ambos carriers ni sumar sus max separados. El reparto de presentación o liquidación no cambia costo físico ni prueba propiedad. Exposición del remolque no se reemplaza por la del tractor.

B2 parcial, B1/E1 abiertos, cero adopciones y 0/4 gates. Siguiente: remuneración/costo patronal y familias fijas en estos mismos perfiles, conservando faltantes de capital y responsabilidad; detener búsquedas genéricas de precios/vidas contables.

Notebook ejecutado con Python estándar, no kernel Jupyter; outputs persistidos. Su vista HTML estática es una reproducción de celdas y outputs, no un viewer interactivo de notebook.